# Cuaderno 1: Modelado Probabilístico del Entorno Real mediante SAREnv
**Proyecto:** Optimización de Estrategias de Búsqueda y Rescate mediante Enjambres de Drones  
**Autor:** Juan Carlos  
**Localización de Estudio:** Parque de la Casa de Campo, Madrid  

Este cuaderno documenta la fase inicial del proyecto: la **inicialización y configuración del entorno de búsqueda**. Utilizando el framework *SAREnv*, transformamos datos geográficos reales en modelos matemáticos de probabilidad de presencia, que servirán como base para la planificación de rutas de rescate.

In [ ]:
import os
import sys

# Ajuste de directorios para asegurar que estamos en la raíz del proyecto
if os.path.basename(os.getcwd()) == 'TFM_JC':
    os.chdir('..')

print(f"Directorio de trabajo: {os.getcwd()}")

## 1. Introducción y Generación del Espacio de Búsqueda

Para delimitar la zona de búsqueda en la Casa de Campo, hemos implementado una **generación basada en un polígono a medida** definido mediante la librería `shapely.geometry.Polygon`, en lugar de utilizar un radio expansivo circular genérico desde un punto central.

### Justificación
La Casa de Campo es un entorno confinado con límites geográficos y operativos estrictos. El uso de un polígono manual permite:
1.  **Recorte Geográfico Exacto:** El sistema utiliza la operación de intersección matemática para descargar exclusivamente los datos de OpenStreetMap que se encuentran dentro de nuestra frontera, evitando procesar zonas colindantes irrelevantes (como núcleos urbanos densos) que introducirían ruido en la misión de rescate.
2.  **Integridad de la MAP:** Asegura que la densidad de probabilidad se distribuya únicamente en el área de interés operativa, cumpliendo con el rigor estadístico necesario para simulaciones SAR reales.

In [ ]:
import os
import shapely
import numpy as np
import osmnx as ox
from sarenv import (
    CLIMATE_TEMPERATE,
    ENVIRONMENT_TYPE_FLAT,
    DataGenerator,
    get_logger,
)
from sarenv.core.loading import DatasetLoader
from sarenv.utils.geo import get_utm_epsg
from pyproj import Transformer

log = get_logger()

# Configuración OSMnx para robustez en la red
ox.settings.timeout = 300  
ox.settings.request_retries = 5  

def generar_casa_de_campo():
    """
    Genera el entorno de la Casa de Campo usando el polígono exacto de Google Earth.
    """
    log.info("--- Iniciando Generación de Entorno (TFM_JC) ---")

    data_gen = DataGenerator()

    polygon_coords = [
        [-3.753046089833776, 40.44343778644211],
        [-3.771067897299357, 40.43808936530237],
        [-3.780967823387276, 40.41887000168707],
        [-3.773618938561301, 40.40203996553611],
        [-3.724145972222016, 40.41588490845162],
        [-3.753046089833776, 40.44343778644211]
    ]

    casa_de_campo_poly = shapely.geometry.Polygon(polygon_coords)
    
    # RUTA ORGANIZADA EN TFM_JC
    output_dir = "TFM_JC/resultados/casa_de_campo"
    os.makedirs(output_dir, exist_ok=True)

    log.info(f"Exportando dataset a {output_dir}...")
    data_gen.export_dataset_from_polygon(
        polygon=casa_de_campo_poly,
        output_directory=output_dir,
        environment_climate=CLIMATE_TEMPERATE, # [Opciones: CLIMATE_TEMPERATE, CLIMATE_DRY]
        environment_type=ENVIRONMENT_TYPE_FLAT, # [Opciones: ENVIRONMENT_TYPE_FLAT, ENVIRONMENT_TYPE_MOUNTAINOUS]
        meter_per_bin=20, # [Opciones recomendadas: 10, 20, 30] (resolución en metros por celda)
    )

    log.info("--- Generación Completada en TFM_JC ---")

# Ejecutamos o cargamos el entorno
output_path = "TFM_JC/resultados/casa_de_campo"
if not os.path.exists(output_path):
    print("Generando entorno de la Casa de Campo...")
    generar_casa_de_campo()
else:
    print(f"El entorno en {output_path} ya está disponible.")

## 2. Sectorización y Resolución Espacial

El proceso de **sectorización** convierte la cartografía vectorial descargada de OpenStreetMap en una cuadrícula discreta (Grid 2D) o mapa de ocupación.

*   **Resolución (`meter_per_bin = 20`):** Se ha configurado una resolución de 20 metros por cada píxel de la matriz.
*   **Impacto Computacional:** Cada celda representa un área de 400 $m^2$. Este valor representa el *equilibrio óptimo* entre la precisión necesaria para capturar senderos y accidentes geográficos, y el coste computacional del simulador, evitando matrices excesivamente densas que comprometan el rendimiento de los algoritmos de planificación en tiempo real.

## 3. Capas e Integración Probabilística

El componente `DataGenerator` de SAREnv extrae múltiples capas de información topográfica. Para consolidarlas en una única **Matriz de Probabilidad (Heatmap)**, el sistema sigue una lógica de integración rigurosa:

1.  **Fusión por Máximos:** Cuando dos elementos (ej. un camino y una zona boscosa) coinciden en una celda, el sistema selecciona el valor de probabilidad más alto mediante la operación `np.maximum`. Esto evita acumulaciones artificiales de probabilidad que no se corresponden con el comportamiento físico real.
2.  **Modelado LPB:** Se aplican pesos específicos basados en el comportamiento de personas perdidas (*Lost Person Behavior*), priorizando infraestructuras lineales y recursos hídricos.
3.  **Exportación:** El resultado final se almacena en un archivo binario `.npy`, normalizado para que el sumatorio de sus elementos sea la unidad, listo para ser consumido por drones autónomos.

In [ ]:
from sarenv import DatasetLoader
import numpy as np

# Carga de la matriz maestra generada para inspección
loader = DatasetLoader(dataset_directory="TFM_JC/resultados/casa_de_campo")
item = loader.load_environment("large") # [Opciones de tamaño de ventana: "small", "medium", "large", "xlarge"]

print(f"Dimensiones de la matriz de probabilidad: {item.heatmap.shape}")
print(f"Probabilidad total acumulada en tamaño 'large': {item.heatmap.sum():.4f}")

## 4. Tabla de Parámetros de Configuración

A continuación se sistematizan los parámetros técnicos definidos para este escenario:

| Variable | Valor Asignado | Descripción Técnica |
| :--- | :--- | :--- |
| `polygon` | Custom (Google Earth) | Perímetro irregular ajustado a la valla real de la Casa de Campo. |
| `environment_climate` | `CLIMATE_TEMPERATE` | Ajusta los pesos de probabilidad según el clima templado del centro de España. |
| `environment_type` | `ENVIRONMENT_TYPE_FLAT` | Define el relieve del terreno (llano) para el cálculo de distancias. |
| `meter_per_bin` | 20 | Resolución espacial: 1 píxel equivale a una celda de 20x20 metros. |
| `size` | `large` | Tamaño de la ventana de búsqueda (Radio de 3.2 km), óptimo para el parque. |
| `num_drones` | 3 | Número de agentes aéreos configurados en la simulación. |
| `budget_meters` | 1,500,000 m | Autonomía de vuelo (presupuesto energético) total del equipo. |

## 4.5. Selector Interactivo de Capas y Perfiles (Demo para el Tutor)

Esta sección añade el **Selector de Capas e Interfaz de Pesos** solicitado por el tutor (Jompy). Permite configurar de forma visual e interactiva:
1.  **Perfil de la víctima:** Carga automáticamente las estadísticas del manual (Autista, Demencia, Senderista) o permite un modo **Personalizado**.
2.  **Pesos de terreno (capas):** Permite cambiar las prioridades de búsqueda de las 10 capas geográficas usando barras deslizantes.
3.  **Radios de dispersión:** Ajustar los percentiles de distancia (Small, Medium, Large, XLarge).

Al hacer clic en **Generar y Visualizar Mapa**, el cuaderno recalcula el mapa de calor combinado de la Casa de Campo en memoria (en menos de 0.1 segundos) y lo dibuja en tiempo real, demostrando el impacto del comportamiento de la víctima en la densidad de búsqueda.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import shapely.geometry
import geopandas as gpd
from sarenv import DataGenerator, CLIMATE_TEMPERATE, ENVIRONMENT_TYPE_FLAT
import sarenv.utils.lost_person_behavior as lpb

# 1. Definir los valores de los perfiles para restablecerlos
AUTISTA_W = {"water": 0.35, "structure": 0.25, "road": 0.18, "woodland": 0.09, "field": 0.09, "brush": 0.04, "scrub": 0.04, "linear": 0.0, "drainage": 0.0, "rock": 0.0}
AUTISTA_R = [0.6, 1.6, 3.7, 15.2]

DEMENCIA_W = {"structure": 0.20, "road": 0.18, "woodland": 0.17, "field": 0.14, "linear": 0.09, "drainage": 0.09, "water": 0.07, "brush": 0.03, "scrub": 0.03, "rock": 0.0}
DEMENCIA_R = [0.3, 1.0, 2.4, 12.8]

SENDERISTA_W = {"linear": 0.25, "field": 0.14, "structure": 0.13, "road": 0.13, "drainage": 0.12, "water": 0.08, "woodland": 0.07, "rock": 0.04, "brush": 0.03, "scrub": 0.02}
SENDERISTA_R = [0.6, 1.8, 3.2, 9.9]

# 2. Cargar el entorno de Casa de Campo previamente generado para evitar re-descargas
polygon_coords = [
    [-3.753046089833776, 40.44343778644211],
    [-3.771067897299357, 40.43808936530237],
    [-3.780967823387276, 40.41887000168707],
    [-3.773618938561301, 40.40203996553611],
    [-3.724145972222016, 40.41588490845162],
    [-3.753046089833776, 40.44343778644211]
]
casa_de_campo_poly = shapely.geometry.Polygon(polygon_coords)

print("Inicializando entorno de la Casa de Campo en memoria (cargando capas de OSM)...")
data_gen = DataGenerator()
master_env = data_gen.generate_environment_from_polygon(casa_de_campo_poly, meter_per_bin=20)
print("¡Entorno inicializado con éxito!")

# 3. Crear los controles de ipywidgets
style = {'description_width': 'initial'}

perfil_dropdown = widgets.Dropdown(
    options=['Autista', 'Demencia', 'Senderista', 'Personalizado'],
    value='Autista',
    description='Perfil de Víctima:',
    style=style
)

# Sliders para pesos de capas
sliders_pesos = {}
features = ["water", "structure", "road", "woodland", "field", "brush", "scrub", "linear", "drainage", "rock"]
features_es = {
    "water": "Agua (water)",
    "structure": "Estructuras (structure)",
    "road": "Carreteras (road)",
    "woodland": "Bosques (woodland)",
    "field": "Campos abiertos (field)",
    "brush": "Malezas (brush)",
    "scrub": "Matorral (scrub)",
    "linear": "Senderos/Líneas (linear)",
    "drainage": "Canales/Drenajes (drainage)",
    "rock": "Rocas (rock)"
}

for feat in features:
    sliders_pesos[feat] = widgets.FloatSlider(
        value=AUTISTA_W[feat],
        min=0.0,
        max=1.0,
        step=0.01,
        description=features_es[feat],
        style=style,
        continuous_update=False
    )

# Sliders para radios de dispersión
sliders_radios = {}
radios_nombres = ["25% (Small)", "50% (Medium)", "75% (Large)", "95% (XLarge)"]
for idx, name in enumerate(radios_nombres):
    sliders_radios[idx] = widgets.FloatSlider(
        value=AUTISTA_R[idx],
        min=0.1,
        max=20.0,
        step=0.1,
        description=f"Radio {name} (km):",
        style=style,
        continuous_update=False
    )

boton_generar = widgets.Button(
    description='Generar y Visualizar Mapa',
    button_style='success',
    tooltip='Pulsa para regenerar el mapa de calor con los pesos actuales',
    icon='refresh'
)

output_plot = widgets.Output()

# 4. Lógicas de actualización reactiva
def actualizar_interfaz_por_perfil(change):
    perfil = change['new']
    if perfil == 'Autista':
        for feat in features:
            sliders_pesos[feat].value = AUTISTA_W[feat]
            sliders_pesos[feat].disabled = True
        for idx in range(4):
            sliders_radios[idx].value = AUTISTA_R[idx]
            sliders_radios[idx].disabled = True
    elif perfil == 'Demencia':
        for feat in features:
            sliders_pesos[feat].value = DEMENCIA_W[feat]
            sliders_pesos[feat].disabled = True
        for idx in range(4):
            sliders_radios[idx].value = DEMENCIA_R[idx]
            sliders_radios[idx].disabled = True
    elif perfil == 'Senderista':
        for feat in features:
            sliders_pesos[feat].value = SENDERISTA_W[feat]
            sliders_pesos[feat].disabled = True
        for idx in range(4):
            sliders_radios[idx].value = SENDERISTA_R[idx]
            sliders_radios[idx].disabled = True
    elif perfil == 'Personalizado':
        for feat in features:
            sliders_pesos[feat].disabled = False
        for idx in range(4):
            sliders_radios[idx].disabled = False

perfil_dropdown.observe(actualizar_interfaz_por_perfil, names='value')

# Desactivar de inicio porque el perfil por defecto es Autista
for feat in features:
    sliders_pesos[feat].disabled = True
for idx in range(4):
    sliders_radios[idx].disabled = True

def click_generar_mapa(b):
    with output_plot:
        clear_output(wait=True)
        print("Regenerando mapa de calor bayesiano en tiempo real...")
        
        pesos_actuales = {feat: sliders_pesos[feat].value for feat in features}
        radios_actuales = [sliders_radios[idx].value for idx in range(4)]
        
        suma_pesos = sum(pesos_actuales.values())
        if suma_pesos > 0:
            pesos_normalizados = {k: v / suma_pesos for k, v in pesos_actuales.items()}
        else:
            pesos_normalizados = {k: 1.0/len(features) for k in pesos_actuales.keys()}
            
        lpb.FEATURE_PROBABILITIES = pesos_normalizados
        lpb.RADIUS_FLAT_TEMPERATE = radios_actuales
        
        feature_heatmap = master_env.get_combined_heatmap()
        mu, sigma = data_gen._lognormal_distribution_estimation(CLIMATE_TEMPERATE, ENVIRONMENT_TYPE_FLAT)
        h, w = feature_heatmap.shape
        y_indices, x_indices = np.mgrid[0:h, 0:w]
        
        centroid = casa_de_campo_poly.centroid
        center_lon, center_lat = centroid.x, centroid.y
        center_point_gdf = gpd.GeoDataFrame(
            geometry=[shapely.Point(center_lon, center_lat)], crs="EPSG:4326"
        )
        center_point_proj = center_point_gdf.to_crs(master_env.projected_crs)
        center_x_world = center_point_proj.geometry.x.iloc[0]
        center_y_world = center_point_proj.geometry.y.iloc[0]
        
        x_coords_world = master_env.xedges[x_indices]
        y_coords_world = master_env.yedges[y_indices]
        dist = np.sqrt((x_coords_world - center_x_world)**2 + (y_coords_world - center_y_world)**2)
        dist_km = np.clip(dist / 1000.0, 1e-6, None)
        
        bell_curve_map = (1 / (dist_km * sigma * np.sqrt(2 * np.pi))) * np.exp(
            -((np.log(dist_km) - mu)**2) / (2 * sigma**2)
        )
        spatial_probability_map = bell_curve_map / bell_curve_map.sum()
        
        # Combinar capas de forma ponderada según deslizadores
        combined_features = np.zeros_like(feature_heatmap)
        for key, individual_heatmap in master_env.heatmaps.items():
            if individual_heatmap is not None:
                combined_features += individual_heatmap.astype(float) * pesos_normalizados.get(key, 0)
        
        feature_heatmap_sum = np.sum(combined_features)
        if feature_heatmap_sum == 0:
            feature_prob_map = np.ones_like(combined_features)
        else:
            feature_prob_map = combined_features / feature_heatmap_sum
            
        combined_map_unnormalized = spatial_probability_map * feature_prob_map
        total_sum = np.sum(combined_map_unnormalized)
        if total_sum > 0:
            final_map = combined_map_unnormalized / total_sum
        else:
            final_map = combined_map_unnormalized
            
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(final_map, cmap='YlOrRd', origin='lower')
        fig.colorbar(im, ax=ax, label='Probabilidad de Presencia')
        ax.set_title(f"Mapa de Calor Regenerativo - Perfil: {perfil_dropdown.value}", fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        
        print(f"¡Mapa de calor actualizado con éxito!")
        print(f"Pesos de terreno utilizados (normalizados):")
        for k, v in pesos_normalizados.items():
            if v > 0.01:
                print(f"  - {features_es[k]}: {v*100:.1f}%")
        print(f"Radios estadísticos utilizados (LKP): {radios_actuales} km")

boton_generar.on_click(click_generar_mapa)

# 5. Organizar en Layouts visuales
caja_perfil = widgets.VBox([perfil_dropdown])

caja_pesos_izq = widgets.VBox([sliders_pesos[f] for f in features[:5]])
caja_pesos_der = widgets.VBox([sliders_pesos[f] for f in features[5:]])
caja_pesos = widgets.HBox([caja_pesos_izq, caja_pesos_der])

caja_radios_izq = widgets.VBox([sliders_radios[0], sliders_radios[1]])
caja_radios_der = widgets.VBox([sliders_radios[2], sliders_radios[3]])
caja_radios = widgets.HBox([caja_radios_izq, caja_radios_der])

acordeon = widgets.Accordion(children=[caja_pesos, caja_radios])
acordeon.set_title(0, 'Ajustes de Pesos por Capa (0.0 - 1.0)')
acordeon.set_title(1, 'Ajustes de Radios Estadísticos (km)')

layout_completo = widgets.VBox([
    widgets.HTML("<h3>Demo Interactiva del Selector de Capas (TFM)</h3>"),
    caja_perfil,
    acordeon,
    widgets.HTML("<br>"),
    boton_generar,
    widgets.HTML("<br>"),
    output_plot
])

display(layout_completo)



## 5. Visualización Geográfica Interactiva (2D)
Generamos la visualización interactiva para confirmar que el modelo de probabilidad se ciñe estrictamente a la geografía de la Casa de Campo proyectada sobre un mapa de satélite/callejero real.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pyproj import Transformer
from sarenv import DatasetLoader, get_logger
from sarenv.utils.geo import get_utm_epsg

log = get_logger()

def verificar_geografia_interactiva():
    output_dir = "TFM_JC/resultados/casa_de_campo"
    
    if not os.path.exists(output_dir):
        log.error(f"No se encuentra la carpeta {output_dir}")
        return

    loader = DatasetLoader(dataset_directory=output_dir)
    item = loader.load_environment("large") 
    
    if not item: return

    features_path = os.path.join(output_dir, "features.geojson")
    boundary_geom = None
    if os.path.exists(features_path):
        with open(features_path, 'r') as f:
            geojson_data = json.load(f)
        for feat in geojson_data['features']:
            if feat['id'] == 'boundary':
                boundary_geom = feat['geometry']
                break

    epsg_code = get_utm_epsg(item.center_point[0], item.center_point[1])
    transformer = Transformer.from_crs(epsg_code, "EPSG:4326", always_xy=True)
    
    heatmap = item.heatmap
    step = 2
    rows, cols = heatmap.shape
    x_coords = np.linspace(item.bounds[0], item.bounds[2], cols)
    y_coords = np.linspace(item.bounds[1], item.bounds[3], rows)
    
    lons, lats, probs = [], [], []
    for r in range(0, rows, step):
        for c in range(0, cols, step):
            p = heatmap[r, c]
            if p > 1e-8:
                lon, lat = transformer.transform(x_coords[c], y_coords[r])
                lons.append(lon)
                lats.append(lat)
                probs.append(p)

    df = pd.DataFrame({'lon': lons, 'lat': lats, 'prob': probs})

    # Usamos density_map (versión moderna) para evitar DeprecationWarnings
    fig = px.density_map(
        df, lat='lat', lon='lon', z='prob', radius=12,
        center=dict(lat=item.center_point[1], lon=item.center_point[0]), 
        zoom=13, map_style="open-street-map", opacity=0.5,
        color_continuous_scale="YlOrRd", 
        range_color=[df['prob'].min(), df['prob'].max()],
        title="VERIFICACIÓN GEOGRÁFICA (TAMAÑO LARGE)"
    )

    if boundary_geom:
        coords = boundary_geom['coordinates'][0]
        # Scattermap para mostrar la línea del polígono
        fig.add_trace(go.Scattermap(
            lon=[c[0] for c in coords], lat=[c[1] for c in coords],
            mode='lines', line=dict(width=2, color='black'), 
            name='Límite Casa de Campo'
        ))

    path_html = os.path.join(output_dir, "verificacion_TFM_JC.html")
    fig.write_html(path_html)
    log.info(f"Mapa interactivo generado en: {path_html}")
    
    # Renderizado directo en el notebook
    fig.show()

verificar_geografia_interactiva()

## 6. Visualización de Densidad de Probabilidad (3D)
Para comprender la topología matemática que procesarán los algoritmos de los drones, renderizamos la matriz de calor como una superficie 3D interactiva, donde la altura (Eje Z) representa la densidad de probabilidad (MAP).

In [ ]:
import os
import numpy as np
import plotly.graph_objects as go
from sarenv import DatasetLoader, get_logger

log = get_logger()

def visualizar_3d_resultados():
    output_dir = "TFM_JC/resultados/casa_de_campo"
    loader = DatasetLoader(dataset_directory=output_dir)
    item = loader.load_environment("large")
    
    if not item: return

    minx, miny, maxx, maxy = item.bounds
    x = np.linspace(minx, maxx, item.heatmap.shape[1])
    y = np.linspace(miny, maxy, item.heatmap.shape[0])

    fig = go.Figure(data=[go.Surface(
        x=x, y=y, z=item.heatmap,
        colorscale='YlOrRd',
        colorbar=dict(title='Probabilidad')
    )])

    fig.update_layout(
        title='Mapa de Probabilidad 3D - Casa de Campo',
        scene=dict(
            xaxis_title='Easting (m)',
            yaxis_title='Northing (m)',
            zaxis_title='Probabilidad'
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    
    path_html = os.path.join(output_dir, "casa_de_campo_3d_interactivo.html")
    fig.write_html(path_html)
    
    fig.show()

visualizar_3d_resultados()

## 7. Ejecución de la Simulación y Evaluación de Estrategias

En esta fase, el sistema utiliza el motor de análisis de **SAREnv** para poner a prueba diferentes algoritmos de cobertura sobre el mapa de la Casa de Campo. El proceso se divide en tres pasos críticos:

1.  **Simulación de Montecarlo:** Se distribuyen estadísticamente 100 víctimas virtuales (survivors) sobre el terreno, priorizando las zonas de mayor probabilidad definidas en la MAP.
2.  **Ejecución de Baselines:** Se ejecutan trayectorias estándar (Espiral, Círculos Concéntricos y Pizza Zig-Zag) para un enjambre de 3 drones con un presupuesto de batería de 1.500 km totales.
3.  **Generación de Resultados:** Se calculan las métricas de rendimiento y se exportan tanto las gráficas comparativas como los mapas de calor con las rutas superpuestas.


In [ ]:
import os
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point
import sarenv
from sarenv.analytics.evaluator import ComparativeEvaluator
from sarenv.utils import plot
from tqdm.auto import tqdm

# 1. Configuración de parámetros REALISTAS (Optimizado para Casa de Campo)
data_dir = "TFM_JC/resultados/casa_de_campo"
base_tfm_dir = Path("TFM_JC/resultados/casa_de_campo")

num_drones = 3
num_lost_persons = 20   # 20 víctimas para un Montecarlo rápido y representativo
budget_meters = 450000  # 450 km: Presupuesto coherente con el área de 12.7 km²
tamaños_a_evaluar = ["large"] 

print("--- Iniciando Evaluación con Parámetros Reales (TFM_JC) ---")

# 2. Inicialización del Evaluador
evaluator = ComparativeEvaluator(
    dataset_directory=data_dir,
    evaluation_sizes=tamaños_a_evaluar,
    num_drones=num_drones,
    num_lost_persons=num_lost_persons,
    budget=budget_meters,
)

# 3. Ejecución de algoritmos base
print("Ejecutando trayectorias (Spiral, Concentric, Pizza)...")
baseline_results, _ = evaluator.run_baseline_evaluations()

# 4. Guardado de gráficas de rendimiento actualizadas
graphs_dir = base_tfm_dir / "graphs"
graphs_dir.mkdir(parents=True, exist_ok=True)
print(f"Exportando gráficas de rendimiento a {graphs_dir}...")
evaluator.plot_results(baseline_results, output_dir=str(graphs_dir))

# 5. Generación de las nuevas rutas (PDF) con el budget ajustado
print("Generando visualizaciones de rutas actualizadas en PDF...")
output_plots_dir = base_tfm_dir / "paths_plots"
output_plots_dir.mkdir(parents=True, exist_ok=True)

for size, env_data in tqdm(evaluator.environments.items(), desc="Mapas"):
    item = env_data["item"]
    data_crs = env_data["crs"]
    center_proj = gpd.GeoDataFrame(geometry=[Point(item.center_point)], crs="EPSG:4326").to_crs(data_crs).geometry.iloc[0]

    for name, generator in tqdm(evaluator.path_generators.items(), desc=f"Rutas {size}", leave=False):
        # El generador ahora usará los 450km de budget total
        generated_paths = generator(center_proj.x, center_proj.y, item.radius_km * 1000, item.heatmap, item.bounds)
        output_file = output_plots_dir / f"{name}_ruta_{size}.pdf"
        
        plot.plot_heatmap(
            item=item, generated_paths=generated_paths,
            name=f"{name} ({size})",
            x_min=item.bounds[0], x_max=item.bounds[2],
            y_min=item.bounds[1], y_max=item.bounds[3],
            output_file=output_file
        )

print(f"--- Proceso Completado. Resultados y PDFs actualizados en {base_tfm_dir} ---")


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pyproj import Transformer
from shapely.geometry import Point
import geopandas as gpd
from sarenv import DatasetLoader, get_logger
from sarenv.utils.geo import get_utm_epsg
from sarenv.analytics.evaluator import ComparativeEvaluator

log = get_logger()

def generar_mapas_por_algoritmo_final_pro():
    output_dir = "TFM_JC/resultados/casa_de_campo"
    # Carpeta específica para los mapas interactivos
    html_plots_dir = os.path.join(output_dir, "path_plots_html")
    os.makedirs(html_plots_dir, exist_ok=True)
    
    loader = DatasetLoader(dataset_directory=output_dir)
    item = loader.load_environment("large")
    if not item: return

    # 1. Configuración de Coordenadas
    data_crs = get_utm_epsg(item.center_point[0], item.center_point[1])
    transformer = Transformer.from_crs(data_crs, "EPSG:4326", always_xy=True)
    
    # 2. Datos del Heatmap
    rows, cols = item.heatmap.shape
    x_coords = np.linspace(item.bounds[0], item.bounds[2], cols)
    y_coords = np.linspace(item.bounds[1], item.bounds[3], rows)
    lons, lats, probs = [], [], []
    for r in range(0, rows, 2): 
        for c in range(0, cols, 2):
            p = item.heatmap[r, c]
            if p > 1e-7:
                lon, lat = transformer.transform(x_coords[c], y_coords[r])
                lons.append(lon); lats.append(lat); probs.append(p)
    
    df_heatmap = pd.DataFrame({'lat': lats, 'lon': lons, 'prob': probs})

    # 3. Evaluador y Colores "Aviation Pro"
    evaluator = ComparativeEvaluator(dataset_directory=output_dir, num_drones=3, budget=450000)
    center_proj = gpd.GeoDataFrame(geometry=[Point(item.center_point)], crs="EPSG:4326").to_crs(data_crs).geometry.iloc[0]
    
    # Paleta técnica: Azul Zafiro, Verde Bosque, Índigo
    colores_drones = ['#0F52BA', '#228B22', '#4B0082'] 
    nombres_colores = ['Azul Zafiro', 'Verde Bosque', 'Índigo']

    # 4. Generación de mapas
    for name, generator in evaluator.path_generators.items():
        print(f"Generando mapa Pro para: {name}...")
        
        fig = px.density_map(
            df_heatmap, lat='lat', lon='lon', z='prob', radius=10,
            center=dict(lat=item.center_point[1], lon=item.center_point[0]),
            zoom=13, map_style="open-street-map", opacity=0.4,
            color_continuous_scale="YlOrRd", title=f"SAR MISSION PLANNING: {name.upper()} STRATEGY"
        )

        fig.update_layout(
            # Colorbar a la izquierda para no estorbar a la leyenda de drones
            coloraxis_colorbar=dict(
                title='Probabilidad',
                thickness=15,
                len=0.5,
                x=-0.12, 
                xanchor='right',
                y=0.5,
                yanchor='middle'
            ),
            legend=dict(
                title="Swarm Fleet (3 Agents)",
                yanchor="top", y=0.99,
                xanchor="right", x=0.99,
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="DarkSlateGray",
                borderwidth=2
            ),
            margin=dict(l=100, r=20, t=60, b=20)
        )

        path_meters = generator(center_proj.x, center_proj.y, item.radius_km * 1000, item.heatmap, item.bounds)
        
        for i, drone_path in enumerate(path_meters):
            path_coords = list(drone_path.coords)
            p_lons, p_lats = [], []
            for cx, cy in path_coords:
                plon, plat = transformer.transform(cx, cy)
                p_lons.append(plon); p_lats.append(plat)
            
            fig.add_trace(go.Scattermap(
                lon=p_lons, lat=p_lats, mode='lines',
                line=dict(width=4, color=colores_drones[i]),
                name=f"Drone {i+1} ({nombres_colores[i]})",
            ))

        # Guardado organizado
        filename = f"mission_{name}_pro.html"
        path_html = os.path.join(html_plots_dir, filename)
        fig.write_html(path_html)
        print(f"-> Mapa guardado en: {path_html}")
        fig.show()

generar_mapas_por_algoritmo_final_pro()


## 8. Métricas de Evaluación de SAREnv
Para cuantificar el éxito de una misión de búsqueda, el framework SAREnv utiliza varias métricas fundamentales que evalúan diferentes dimensiones del rendimiento de los algoritmos:

1.  **Likelihood Score (APOD - Accumulated Probability of Detection):**
    Representa la probabilidad de detección acumulada. Mide qué porcentaje de la "masa de probabilidad" total del mapa ha sido inspeccionado por los drones. Es la métrica de eficacia total.

2.  **Time Discounted Score (TDPD - Time-Discounted Probability of Detection):**
    Es similar a la anterior, pero penaliza los hallazgos tardíos mediante un factor de descuento temporal. Refleja la realidad operativa de que encontrar a una víctima en la primera hora es mucho más valioso que encontrarla en la décima.

3.  **Victims Found % (LPDS - Lost Person Discovery Score):**
    Indica el éxito empírico mediante el método de Montecarlo. El sistema "siembra" 100 víctimas virtuales siguiendo la densidad del mapa de calor y calcula cuántas de ellas habrían sido detectadas por la ruta de los drones.

4.  **Area Covered (km²):**
    Métrica puramente física que indica la superficie neta del terreno que ha sido escaneada por la huella de los sensores de los drones.

5.  **Total Path Length (km):**
    Mide la distancia total recorrida por el enjambre de drones. Es fundamental para evaluar la eficiencia energética y asegurar que las rutas planificadas no excedan el presupuesto de batería (*budget*) asignado.


In [ ]:
import os
from IPython.display import Image, display

# Ruta a la carpeta de gráficas generadas en la evaluación
graphs_path = "TFM_JC/resultados/casa_de_campo/graphs"

# Lista de gráficas clave para mostrar
archivos_graficas = [
    "plot_likelihood_score.png",
    "plot_time-discounted_score.png",
    "plot_victims_found_.png",
    "plot_area_covered_(km²).png",
    "plot_total_path_length_(km).png",
]

print("Visualizando resultados estadísticos de la simulación en Casa de Campo:")

for archivo in archivos_graficas:
    full_path = os.path.join(graphs_path, archivo)
    if os.path.exists(full_path):
        print(f"\nMostrando: {archivo}")
        display(Image(filename=full_path))
    else:
        print(f"\nLa gráfica '{archivo}' no se encontró. Asegúrate de haber ejecutado 'evaluar_casa_de_campo.py'.")

## 9. Referencias Bibliográficas (Base Teórica)

*   **Koester, R. J. (2008).** *Lost Person Behavior: A search and rescue guide on where to look - for land, air and water*. dbS Productions. (Base para los pesos de probabilidad topológica).
*   **Doherty, K., et al. (2014).** *A probabilistic framework for search and rescue operations*. (Fundamento de las distribuciones matemáticas de probabilidad).

### Preguntas Jompy

#### 1. ¿Cómo se genera el espacio de búsqueda inicial?
Se puede generar de dos maneras utilizando el generador `DataGenerator`:
*   **A partir de un punto central (Circular):** Se define un `center_point` y un radio operativo en kilómetros. El sistema genera un búfer circular de radio `radius_km` alrededor del punto. El radio se selecciona según la escala de la misión (`small`, `medium`, `large` o `xlarge`), la cual corresponde a los percentiles de distancia que suele recorrer una persona extraviada.
*   **A partir de un polígono personalizado (Caso Casa de Campo):** Se define una frontera geográfica exacta (`shapely.geometry.Polygon`). En `export_dataset_from_polygon`, el sistema calcula el centroide de dicho polígono y descarga mediante `osmnx` únicamente las características geográficas de OpenStreetMap (OSM) que intersectan con este polígono de búsqueda, evitando ruido operativo y recortando el mapa de forma exacta.

#### 2. ¿Cómo sectoriza SAREnv el espacio de búsqueda?
El proceso de sectorización es la discretización del espacio geográfico continuo en una rejilla o mapa de celdas bidimensional (Grid 2D):
*   **Proyección de Coordenadas:** Primero, las coordenadas geográficas (latitud/longitud en `EPSG:4326`) se transforman dinámicamente a coordenadas planas UTM (proyectadas en metros) utilizando la zona local correspondiente (p. ej., huso 30N para Madrid).
*   **Resolución Espacial (`meter_per_bin`):** Define el tamaño físico en metros del lado de cada celda de la matriz (p. ej., 20 metros por celda).
*   **Dimensionamiento del Grid:** Las dimensiones de la matriz ($N_{\text{filas}} \times M_{\text{columnas}}$) se calculan dividiendo la anchura y la altura de la caja delimitadora (Bounding Box) del polígono proyectado por el parámetro `meter_per_bin`.

#### 3. ¿Cuáles son las capas de información que integra y cómo describen la probabilidad?
SAREnv clasifica la información topográfica descargada de OSM en varias capas de características y las integra siguiendo un modelo matemático de probabilidad de presencia:
*   **Capas de Información:** Se extraen 10 capas temáticas con pesos empíricos basados en estadísticas de comportamiento de personas perdidas (*Lost Person Behavior*), definidos en `FEATURE_PROBABILITIES`:
    *   `linear` (0.25): Vías férreas, vallas, muros (el peso más alto debido a que las personas tienden a seguir líneas guía).
    *   `field` (0.14) y `road` / `structure` (0.13): Campos, prados, carreteras, senderos, edificaciones.
    *   `drainage` (0.12) y `water` (0.08): Canales, zanjas, ríos, lagos (atracción por agua/recursos).
    *   `woodland` (0.07), `rock` (0.04), `scrub` (0.03), `brush` (0.02): Bosques, rocas, matorrales y césped.
*   **Fusión Matemática de Probabilidad (MAP):**
    *   **Fusión por Máximos (Espacial):** En `get_combined_heatmap`, cuando hay solapamiento de capas en una misma celda del grid, se aplica la función `np.maximum` entre capas ponderadas. Esto evita acumulaciones artificiales de probabilidad (ya que la presencia de dos características no duplica la presencia real de la víctima).
    *   **Ponderación por Distancia (Log-normal):** En `_lognormal_distribution_estimation`, el sistema estima los parámetros $\mu$ y $\sigma$ de una curva de distribución log-normal sobre la distancia radial al punto de planificación inicial (o centroide).
    *   **Multiplicación y Normalización:** Se multiplica el mapa de características por el de distribución de distancia, y se divide por la suma total para que la matriz final normalizada sume exactamente 1.0.